# Phase 5: Model Training, Evaluation & Tuning

This notebook implements a clean train/test split with strict prevention of data leakage. It builds preprocessing and classification pipelines for 4 models (Logistic Regression, KNN, Random Forest, and XGBoost/Gradient Boosting), measures wall-clock training time, performs hyperparameter tuning via GridSearchCV on the top-performing architecture, evaluates performance using standard classification metrics, and serializes the best model.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import time
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)

try:
    from xgboost import XGBClassifier
    xgb_clf = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    print("Using XGBoost Classifier")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    xgb_clf = GradientBoostingClassifier(random_state=42)
    print("Using Scikit-Learn Gradient Boosting Classifier (Fallback)")

In [ ]:
conn = sqlite3.connect('../data/cardio_cleaned.db')
df = pd.read_sql_query('SELECT * FROM cardio', conn)
conn.close()
df.shape

In [ ]:
X = df.drop(columns=['cardio'])
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 1. Pipeline Workflows

We define a preprocessor using `ColumnTransformer` to handle scaling of continuous variables and one-hot encoding of categorical features. To prevent data leakage, the preprocessor will be fitted only on the training set as part of the model pipelines.

In [ ]:
num_features = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi', 'pulse_pressure', 'age_cholesterol_risk']
cat_features = ['cholesterol', 'gluc']
bin_features = ['gender', 'smoke', 'alco', 'active']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features),
        ('bin', 'passthrough', bin_features)
    ]
)

In [ ]:
pipelines = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier(n_neighbors=5))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', xgb_clf)
    ])
}

## 2. Model Benchmarking

We train all 4 baseline models and log their wall-clock training times and performance metrics.

In [ ]:
results = {}
for name, pipeline in pipelines.items():
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'Training Time (s)': train_time,
        'Pipeline': pipeline
    }

bench_df = pd.DataFrame(results).T.drop(columns=['Pipeline'])
bench_df

## 3. Hyperparameter Tuning

We apply automated optimization via GridSearchCV to tune the hyperparameters of the XGBoost/Gradient Boosting model.

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1]
}
grid_search = GridSearchCV(pipelines['XGBoost'], param_grid, cv=3, scoring='f1', n_jobs=-1)
start_time = time.time()
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start_time
print("Best parameters:", grid_search.best_params_)
print("Tuning completed in:", tuning_time, "seconds")

In [ ]:
best_xgb = grid_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)
y_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]

tuned_results = {
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned),
    'Recall': recall_score(y_test, y_pred_tuned),
    'F1-score': f1_score(y_test, y_pred_tuned),
    'ROC-AUC': roc_auc_score(y_test, y_prob_tuned),
    'Training Time (s)': tuning_time
}
print("Tuned Model Performance:")
for metric, value in tuned_results.items():
    print(f"{metric}: {value:.4f}")

## 4. Evaluation Metrics & Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, res) in zip(axes.ravel(), results.items()):
    pipe = res['Pipeline'] if name != 'XGBoost' else best_xgb
    pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
for name, res in results.items():
    pipe = res['Pipeline'] if name != 'XGBoost' else best_xgb
    prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.show()

## 5. Model Serialization

We save the tuned model to `/models/best_cardio_model.joblib`.

In [ ]:
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'best_cardio_model.joblib')
joblib.dump(best_xgb, model_path)
print("Best model serialized successfully to:", model_path)